# xlingual-drift: Colab runner

GPU-only steps run here: serving open-weight models with vLLM, and later the SFT ablation.
Everything else (environment, verifier, stats) is developed locally.

Workflow: clone from GitHub each session, write raw logs to Drive (they survive
disconnects), commit condensed results back to GitHub before the session ends.
Runs are resumable, so after a disconnect just rerun from the top.

In [ ]:
# 1. Which GPU did Colab give us? The run config decides whether it is big enough (cell 5).
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Clone or update the repo. The repo is private, so this uses a GitHub token
#    stored in Colab Secrets as GITHUB_TOKEN (key icon in the left sidebar).
import os
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists("/content/xlingual-drift"):
    !git clone -q https://{token}@github.com/khalerishabh/xlingual-drift.git /content/xlingual-drift
else:
    !cd /content/xlingual-drift && git pull -q
%cd /content/xlingual-drift
!git log --oneline -1

In [ ]:
# 3. Install. The harness needs only light packages, in Colab's own Python.
#    vLLM goes into a separate venv so it never touches Colab's preinstalled
#    torch/torchaudio/RAPIDS; --torch-backend=auto picks the PyTorch build that
#    matches this machine's GPU driver. The vLLM version is pinned by the run config,
#    because it is part of what defines a backbone.
!pip install -q -r requirements.txt
!pip install -q uv
import yaml
CONFIG = "configs/capability_gate_v2.yaml"
VLLM_VERSION = yaml.safe_load(open(CONFIG))["serving"]["vllm_version"]
if not os.path.exists("/content/vllm-env/bin/vllm"):
    !uv venv -q /content/vllm-env
    !uv pip install -q -p /content/vllm-env/bin/python "vllm=={VLLM_VERSION}" --torch-backend=auto
# ninja: vLLM compiles some kernels at warm-up (e.g. the FP8 fallback on A100) and needs it.
!uv pip install -q -p /content/vllm-env/bin/python ninja
!/content/vllm-env/bin/python -c "import torch, vllm; print('torch', torch.__version__, '| CUDA build', torch.version.cuda, '| GPU visible', torch.cuda.is_available(), '| vllm', vllm.__version__)"

In [ ]:
# 4. Raw logs go to Drive so a disconnect never loses finished episodes.
from google.colab import drive
drive.mount("/content/drive")
RUNS_DIR = "/content/drive/MyDrive/xlingual-drift-runs"
os.makedirs(RUNS_DIR, exist_ok=True)

## Start the model server

The model, weights revision, parsers and minimum GPU memory come from the run config's
`serving` block, so the notebook can only serve what the config pins. Gate v2 uses the
official **Qwen3.6-27B-FP8**, which fits a 40GB or 80GB A100. On a smaller GPU the cell stops
and asks you to switch runtime. The first start downloads the weights (roughly 30GB).
Server output goes to `/content/vllm.log`.

In [ ]:
# 5. Serve exactly the backbone the run config pins, and record what was served.
#    A backbone is the model plus its exact weights and precision; run_grid
#    refuses to mix episodes from different setups in one log.
import json, subprocess, time, urllib.request, yaml
from huggingface_hub import HfApi

serving = yaml.safe_load(open(CONFIG))["serving"]

gpu_name, vram_mib = [x.strip() for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.split(",")]
vram_gb = int(vram_mib) / 1024
if vram_gb < serving["min_vram_gb"]:
    raise SystemExit(f"{serving['model']} needs >= {serving['min_vram_gb']} GB of GPU memory; this runtime has "
                     f"{gpu_name} ({vram_gb:.0f} GB). Disconnect, choose A100, and reconnect.")

sha = HfApi().model_info(serving["model"], revision=serving["revision"]).sha
vllm_version = subprocess.run(["/content/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                              capture_output=True, text=True).stdout.strip()
if vllm_version != serving["vllm_version"]:
    raise SystemExit(f"vLLM {vllm_version} installed but the config pins {serving['vllm_version']}. "
                     "Delete /content/vllm-env and rerun cell 3.")
SERVING_JSON = "/content/serving.json"
json.dump({"model": serving["model"], "revision": sha, "vllm_version": vllm_version,
           "gpu_name": gpu_name, "vram_gb": round(vram_gb), "max_model_len": serving["max_model_len"]},
          open(SERVING_JSON, "w"), indent=2)
print(open(SERVING_JSON).read())

cmd = ["/content/vllm-env/bin/vllm", "serve", serving["model"],
       "--revision", sha,
       "--port", "8000",
       "--max-model-len", str(serving["max_model_len"]),
       "--gpu-memory-utilization", "0.90",
       "--enable-auto-tool-choice",
       "--tool-call-parser", serving["tool_call_parser"],
       "--reasoning-parser", serving["reasoning_parser"],
       *serving.get("extra_args", [])]
import os
# The venv's bin must be on PATH so vLLM finds build tools (ninja) it installed there.
env = {**os.environ, "PATH": "/content/vllm-env/bin:" + os.environ["PATH"]}
server = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT, env=env)

for i in range(240):
    try:
        if urllib.request.urlopen("http://localhost:8000/health", timeout=5).status == 200:
            print(f"server up after ~{i * 10}s")
            break
    except Exception:
        pass
    if server.poll() is not None:
        raise RuntimeError("vLLM exited; see the tail of /content/vllm.log below")
    time.sleep(10)
else:
    raise TimeoutError("server not up after 40 min")

In [ ]:
# If the server failed to start, the reason is here.
!tail -n 30 /content/vllm.log

## Checks before spending GPU time

In [ ]:
# 6. Harness sanity: tests and the mock smoke test (no model involved).
!python -m pytest -q
!python -m eval.run_grid --config configs/smoke_test.yaml --out /content/smoke_test.jsonl | tail -2

## Capability gate v2 (late-binding templates)

270 episodes: 3 templates x horizons 2-12 x 5 languages x 3 seeds, thinking on.
16 concurrent episodes; vLLM batches them. Safe to rerun after a disconnect: `--resume`
skips finished episodes.

In [ ]:
# 7. Run the gate. --serving stamps every episode with what was served, and refuses to
#    resume into a log produced under a different model, revision or vLLM version.
LOG = f"{RUNS_DIR}/capability_gate_v2_qwen36_27b_fp8.jsonl"
!python -m eval.run_grid --config {CONFIG} --serving {SERVING_JSON} --resume --workers 16 --out {LOG}

In [ ]:
# 8. Early vs late constraints by language x horizon (clean episodes, 9 per cell).
import json, collections
from eval.metrics import latest_scored, rate_table

records = latest_scored([json.loads(l) for l in open(LOG, encoding="utf-8")])
print(f"{len(records)} scored episodes")
langs, hs = ["en", "zh", "hi", "ta", "hinglish"], [2, 4, 6, 8, 10, 12]
views = (("success", "full success", False), ("flight_correct", "early constraints (flight)", False),
         ("all_late_survived", "late constraints (seat + meal), all episodes", False),
         ("all_late_survived", "late constraints (seat + meal), booked episodes only", True))
for field, label, only_booked in views:
    table = rate_table(records, field, only_booked)
    print(f"\n{label}")
    print("  lang      " + "  ".join(f"h{h:<3}" for h in hs))
    for lang in langs:
        print(f"  {lang:<9} " + "  ".join(f"{table[(lang, h)][0]:.2f}" if (lang, h) in table else " -- " for h in hs))

tok = [r["transcript"]["usage"]["completion_tokens"] for r in records]
print(f"\nmean completion tokens per episode: {sum(tok) / len(tok):.0f}")

In [ ]:
# 9. Failures: what was booked vs what was asked.
for r in sorted(records, key=lambda r: (r["template_id"], r["language"], r["horizon"], r["seed"])):
    if not r["success"]:
        err = next((s["error"]["message"] for s in r["steps"] if "error" in s), "")
        print(f'{r["template_id"]} {r["language"]:<9} h{r["horizon"]:<2} s{r["seed"]} flight={r["booked_flight"]} '
              f'seat={r["booked_seat_type"]} meal={r["booked_meal"]} steps={r["num_steps_taken"]} '
              f'limit={r["hit_step_limit"]} first_err={err[:60]}')

## Get the log to your computer

Download the raw log and put it in `C:\\Rishabh\\xlingual-drift\\runs\\` so it can be analysed
locally. Then disconnect: Runtime -> Disconnect and delete runtime.

In [ ]:
# 10. Download the raw log.
from google.colab import files
files.download(LOG)